In [73]:
%matplotlib inline
import torch
import torchvision
from torch.utils import data
from torchvision import transforms
from IPython import display


In [74]:
import IPython
print(IPython.__version__) # 作为jupyter的依赖自动安装好了

9.15.0


In [75]:
def get_dataloader_workers():
    """使用4个进程读取数据"""
    return 0

In [76]:
# 整合所有的组件, 获取FashionMNIST数据集，返回训练集和验证集的数据迭代器

def load_data_fashion_mnist(batch_size, resize=None):
    """下载FashionMNIST的数据集，然后将其加载到内存中"""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True
    ) 
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True
    ) 

    return (data.DataLoader(mnist_train, batch_size, shuffle=True, 
                num_workers=get_dataloader_workers()),
            data.DataLoader(mnist_train, batch_size, shuffle=True,
                num_workers=get_dataloader_workers()))

In [77]:
batch_size = 32
train_iter, test_iter = load_data_fashion_mnist(batch_size, resize=None)

In [78]:
num_inputs = 784
num_outputs = 10

In [79]:
display.display("hello")

'hello'

In [80]:
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)
W, W.shape, b, b.shape

(tensor([[ 3.5076e-03, -1.1609e-02,  7.7738e-03,  ...,  4.4282e-05,
          -2.1584e-02,  1.0059e-02],
         [ 6.4848e-03, -4.4649e-03, -8.0784e-04,  ...,  9.3795e-03,
           1.6315e-02, -4.2145e-03],
         [ 4.9446e-03,  2.8205e-03,  2.0859e-03,  ...,  1.5293e-02,
           6.1973e-03, -1.2433e-02],
         ...,
         [-8.6220e-03,  1.0221e-02,  7.0228e-03,  ..., -4.6642e-03,
          -3.4441e-03, -1.5738e-02],
         [-5.4135e-03,  7.8232e-03,  7.1152e-03,  ...,  2.1009e-03,
          -6.5976e-03, -9.8824e-03],
         [-1.0804e-02,  7.9473e-03, -5.7950e-03,  ..., -9.3758e-03,
          -1.8444e-02,  1.2096e-03]], requires_grad=True),
 torch.Size([784, 10]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True),
 torch.Size([10]))

In [81]:
# 定义softmax操作

X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
X.sum(0, keepdim=True), X.sum(1, keepdim=True)

(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

In [82]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition # 这里使用了广播机制

In [83]:
X = torch.normal(0, 1, (2, 5))
print(X)
X_prob = softmax(X)
print(X_prob, X_prob.sum(1), sep='\n')

tensor([[ 0.5080, -0.5523, -0.0470, -0.8722, -0.8129],
        [-0.6296,  0.7421, -0.7297,  0.5806,  0.1667]])
tensor([[0.4100, 0.1420, 0.2354, 0.1031, 0.1094],
        [0.0876, 0.3452, 0.0792, 0.2938, 0.1942]])
tensor([1.0000, 1.0000])


In [84]:
def net(X):
    return softmax(torch.matmul(X.reshape(-1, W.shape[0]), W) + b)

In [85]:
# 定义损失函数
y = torch.tensor([0, 2])
print(y)
y_hat = torch.tensor([[0.1, 0.3, 0.6],[0.3, 0.2, 0.5]])
y_hat[[0, 1], y]

tensor([0, 2])


tensor([0.1000, 0.5000])

In [86]:
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(0, len(y_hat)), y])

cross_entropy(y_hat, y)

tensor([2.3026, 0.6931])

In [87]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1: 
        # 确保第一轴长度大于1，第二轴长度大于1
        y_hat = y_hat.argmax(axis = 1) # 得到每次预测的最大的概率的值的索引
    cmp = y_hat.type(y.dtype) == y
    # print(cmp)
    return float(cmp.type(y.dtype).sum())

In [88]:
accuracy(y_hat, y) / len(y)

0.5

In [89]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [90]:
# 评估任意模型的精度

def evaluate_accuracy(net, data_iter):
    """计算指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval() # 将模型设置为评估模式
    metric = Accumulator(2) #正确预测数，预测总数
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel())

    return metric[0] / metric[1]

In [91]:
evaluate_accuracy(net, test_iter)

0.06755

In [ ]:
# 训练
def train_epoch_ch3(net, train_iter, loss, updater):
    """训练模型一轮"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    